In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from scripts.data_loader import (
    load_proact_alsfrs,
    load_proact_demographics,
    load_proact_als_history,
    load_proact_deathdata,
    load_proact_treatment,
    load_targetals_agri_clinical
)
from scripts.preprocessing import standardize_columns

In [ ]:
alsfrs = load_proact_alsfrs()
demo = load_proact_demographics()
hist = load_proact_als_history()
death = load_proact_deathdata()
treat = load_proact_treatment()
agri = load_targetals_agri_clinical()

datasets = {
    "PROACT_ALSFRS": alsfrs,
    "PROACT_DEMOGRAPHICS": demo,
    "PROACT_ALSHISTORY": hist,
    "PROACT_DEATHDATA": death,
    "PROACT_TREATMENT": treat,
    "TARGETALS_AGRI_CLINICAL": agri
}

In [ ]:
for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

In [ ]:
alsfrs_s = standardize_columns(alsfrs)
demo_s = standardize_columns(demo)
hist_s = standardize_columns(hist)
death_s = standardize_columns(death)
treat_s = standardize_columns(treat)
agri_s = standardize_columns(agri)

datasets_s = {
    "PROACT_ALSFRS": alsfrs_s,
    "PROACT_DEMOGRAPHICS": demo_s,
    "PROACT_ALSHISTORY": hist_s,
    "PROACT_DEATHDATA": death_s,
    "PROACT_TREATMENT": treat_s,
    "TARGETALS_AGRI_CLINICAL": agri_s
}

In [ ]:
for name, df in datasets_s.items():
    print("=" * 80)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

In [ ]:
for name, df in datasets_s.items():
    print("=" * 80)
    print(name)
    display(df.head(3))

In [ ]:
import pandas as pd

summary_rows = []

for name, df in datasets_s.items():
    summary_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "unique_subject_id": df["subject_id"].nunique() if "subject_id" in df.columns else None,
        "missing_subject_id": int(df["subject_id"].isna().sum()) if "subject_id" in df.columns else None,
        "total_missing_cells": int(df.isna().sum().sum())
    })

summary_df = pd.DataFrame(summary_rows).sort_values("rows", ascending=False)
summary_df

In [ ]:
for name, df in datasets_s.items():
    print("=" * 80)
    print(name)
    missing_summary = (
        df.isna().mean()
        .sort_values(ascending=False)
        .head(15)
        .rename("missing_fraction")
        .reset_index()
    )
    missing_summary.columns = ["column", "missing_fraction"]
    display(missing_summary)

In [ ]:
patient_col = "subject_id"

for name, df in datasets_s.items():
    if patient_col in df.columns:
        print("=" * 80)
        print(name)
        print("Rows:", len(df))
        print("Unique patients:", df[patient_col].nunique())
        print("Missing patient IDs:", df[patient_col].isna().sum())
        print("Duplicate patient ID rows:", df.duplicated(subset=[patient_col]).sum())

In [ ]:
visit_counts = (
    alsfrs_s[patient_col]
    .value_counts()
    .rename("num_rows_per_patient")
    .reset_index()
)

visit_counts.columns = [patient_col, "num_rows_per_patient"]
visit_counts.describe()

In [ ]:
visit_counts.sort_values("num_rows_per_patient", ascending=False).head(15)

In [ ]:
alsfrs_s.columns.tolist()

In [ ]:
time_col = "alsfrs_delta"

alsfrs_s[[patient_col, time_col]].head(10)

In [ ]:
print("Time column dtype:", alsfrs_s[time_col].dtype)
print("Missing values:", alsfrs_s[time_col].isna().sum())
print("Unique values:", alsfrs_s[time_col].nunique())

In [ ]:
sample_patients = visit_counts.sort_values("num_rows_per_patient", ascending=False).head(3)[patient_col].tolist()
sample_patients

In [ ]:
for pid in sample_patients:
    print("=" * 80)
    print("Patient:", pid)
    display(
        alsfrs_s.loc[alsfrs_s[patient_col] == pid, [patient_col, time_col]].head(20)
    )

In [ ]:
[col for col in alsfrs_s.columns if "alsfr" in col.lower()]

In [ ]:
target_col = "alsfrs_r_total"

alsfrs_s[[patient_col, time_col, target_col]].head(10)

In [ ]:
alsfrs_s[[patient_col, time_col, target_col]].isna().mean()

In [ ]:
demo_s.columns.tolist()

In [ ]:
demo_s.head(10)

In [ ]:
hist_s.columns.tolist()

In [ ]:
hist_s.head(10)

In [ ]:
treat_s.columns.tolist()

In [ ]:
death_s.columns.tolist()

In [ ]:
agri_s.columns.tolist()

In [ ]:
agri_s.head(10)

In [ ]:
planning_rows = [
    {
        "dataset": "PROACT_ALSFRS",
        "likely_role": "longitudinal outcome table",
        "patient_id_column": patient_col,
        "time_column": time_col,
        "target_column": target_col
    },
    {
        "dataset": "PROACT_DEMOGRAPHICS",
        "likely_role": "static baseline table",
        "patient_id_column": "subject_id" if "subject_id" in demo_s.columns else None,
        "time_column": None,
        "target_column": None
    },
    {
        "dataset": "PROACT_ALSHISTORY",
        "likely_role": "baseline or clinical history table",
        "patient_id_column": "subject_id" if "subject_id" in hist_s.columns else None,
        "time_column": None,
        "target_column": None
    },
    {
        "dataset": "PROACT_TREATMENT",
        "likely_role": "possible repeated treatment table",
        "patient_id_column": "subject_id" if "subject_id" in treat_s.columns else None,
        "time_column": None,
        "target_column": None
    },
    {
        "dataset": "PROACT_DEATHDATA",
        "likely_role": "survival outcome or follow-up table",
        "patient_id_column": "subject_id" if "subject_id" in death_s.columns else None,
        "time_column": None,
        "target_column": None
    },
    {
        "dataset": "TARGETALS_AGRI_CLINICAL",
        "likely_role": "external clinical comparison or future integration table",
        "patient_id_column": "subject_id" if "subject_id" in agri_s.columns else None,
        "time_column": None,
        "target_column": None
    }
]

planning_df = pd.DataFrame(planning_rows)
planning_df

In [ ]:
print("Planned modeling structure:")
print("- One row will represent one patient visit.")
print("- PROACT_ALSFRS will serve as the core longitudinal table.")
print("- The primary target variable will be alsfrs_r_total.")
print("- The visit ordering variable will be alsfrs_delta.")
print("- Static features from demographics and ALS history can later be merged by subject_id.")
print("- Treatment and death data may be incorporated later depending on model scope.")
print("- Target ALS AGRI clinical data will require separate variable alignment before integration.")

In [ ]:
alsfrs_core = alsfrs_s[[patient_col, time_col, target_col]].copy()
alsfrs_core.head()

In [ ]:
alsfrs_core.shape

In [ ]:
alsfrs_core = alsfrs_core.sort_values([patient_col, time_col]).reset_index(drop=True)
alsfrs_core.head(10)

In [ ]:
print("Missingness in core longitudinal table:")
print(alsfrs_core.isna().mean())

print("\nUnique patients:", alsfrs_core[patient_col].nunique())
print("Rows:", len(alsfrs_core))

In [ ]:
output_path = repo_root / "data" / "interim" / "alsfrs_core.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

alsfrs_core.to_csv(output_path, index=False)